##### Copyright 2026 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Social Media Intelligence with XPOZ MCP

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Social_media_intelligence_with_XPOZ_MCP.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook shows how to use [XPOZ](https://xpoz.ai) — a social-intelligence MCP server with 1.5B+ indexed posts across Twitter, Instagram, Reddit & TikTok — to build a real-time social media analysis pipeline with the Gemini API.

You will learn to:
- Connect to a remote MCP server via HTTP (JSON-RPC)
- Fetch cross-platform social data (Twitter, Reddit, Instagram)
- Use Gemini to produce structured sentiment analysis from raw social posts

## Setup

### Install the SDK

In [2]:
%pip install -U -q 'google-genai>=1.0.0' httpx


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /Users/oferweintraub/OferW/xpoz-cookbooks/.venv/bin/python3 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


### Set up your API keys

You need two API keys stored as [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb):
- `GOOGLE_API_KEY` — from [Google AI Studio](https://aistudio.google.com/)
- `XPOZ_API_KEY` — free at [xpoz.ai](https://xpoz.ai) (100K results/month)

In [3]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
XPOZ_API_KEY = userdata.get("XPOZ_API_KEY")

### Choose a model

In [4]:
MODEL_ID="gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

In [5]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

## Connect to XPOZ MCP server

XPOZ provides social data via the [Model Context Protocol](https://modelcontextprotocol.io/). The server accepts JSON-RPC requests over HTTP. Create a lightweight client that initializes a session and calls tools like `countTweets` and `getTwitterPostsByKeywords`.

In [6]:
import json
import re
import csv
import io
import httpx
from datetime import datetime, timedelta

XPOZ_URL = "https://mcp.xpoz.ai/mcp"
_msg_id = 0
_session_id = None

In [7]:
# @title Helper functions for MCP communication
def _parse_sse(text):
    """Extract JSON-RPC result from an SSE response."""
    result = {}
    for line in text.split("\n"):
        if line.startswith("data: "):
            try:
                result = json.loads(line[6:])
            except json.JSONDecodeError:
                pass
    return result


async def _mcp_post(http_client, payload):
    """POST a JSON-RPC request and handle JSON or SSE responses."""
    global _session_id
    headers = {
        "Authorization": f"Bearer {XPOZ_API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if _session_id:
        headers["Mcp-Session-Id"] = _session_id
    resp = await http_client.post(XPOZ_URL, json=payload, headers=headers)
    if "mcp-session-id" in resp.headers:
        _session_id = resp.headers["mcp-session-id"]
    ct = resp.headers.get("content-type", "")
    if "text/event-stream" in ct:
        return _parse_sse(resp.text)
    elif resp.text.strip():
        return resp.json()
    return {}

In [8]:
async def _ensure_session(http_client):
    """Initialize the MCP session if not already active."""
    global _session_id
    if _session_id:
        return
    await _mcp_post(http_client, {
        "jsonrpc": "2.0", "id": 0, "method": "initialize",
        "params": {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "cookbook", "version": "1.0"},
        },
    })
    await _mcp_post(http_client, {
        "jsonrpc": "2.0", "method": "notifications/initialized",
    })

In [9]:
# @title Parser for XPOZ compact tabular format
def _parse_xpoz(text):
    """Parse XPOZ compact tabular format into a Python dict."""
    result = {"success": "success: true" in text, "data": {}}
    hdr = re.search(r"results\[\d+\]\{([^}]+)\}:", text)
    if hdr:
        fields = hdr.group(1).split(",")
        rows = []
        for line in text[hdr.end():].split("\n"):
            if not line.startswith("    "):
                if rows or (line.strip() and not line.startswith(" ")):
                    break
                continue
            try:
                vals = next(csv.reader(io.StringIO(line.strip())))
                row = {}
                for i, f in enumerate(fields):
                    if i < len(vals):
                        v = vals[i].strip()
                        if v in ("null", ""):
                            row[f] = None
                        else:
                            try:
                                row[f] = int(v)
                            except ValueError:
                                try:
                                    row[f] = float(v)
                                except ValueError:
                                    row[f] = v
                rows.append(row)
            except Exception:
                pass
        result["data"]["results"] = rows
    for m in re.finditer(r"^\s{2}(\w+):\s+(.+)$", text, re.MULTILINE):
        k, v = m.group(1), m.group(2).strip()
        if k.startswith("results"):
            continue
        try:
            result["data"][k] = int(v)
        except ValueError:
            try:
                result["data"][k] = float(v)
            except ValueError:
                result["data"][k] = v
    return result

In [10]:
async def call_xpoz(tool_name, params):
    """Call an XPOZ MCP tool and return parsed results."""
    global _msg_id
    _msg_id += 1
    async with httpx.AsyncClient(timeout=120) as http_client:
        await _ensure_session(http_client)
        data = await _mcp_post(http_client, {
            "jsonrpc": "2.0", "id": _msg_id,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": params},
        })
    if "error" in data:
        print(f"MCP error: {data['error']}")
        return {"data": {"results": []}}
    if "result" not in data:
        print(f"Unexpected response: {str(data)[:200]}")
        return {"data": {"results": []}}
    content = data.get("result", {}).get("content", [{}])
    text = content[0].get("text", "{}") if content else "{}"
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return _parse_xpoz(text)

Verify the connection works.

In [11]:
def days_ago(n):
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")

test = await call_xpoz("countTweets", {"phrase": "Google DeepMind", "startDate": days_ago(1)})
print(f"Connected to XPOZ MCP — Google DeepMind tweets in last 24h: {test['data'].get('count', 'ok'):,}")


Connected to XPOZ MCP — Google DeepMind tweets in last 24h: 11,465


## Fetch social data

Analyze **Google DeepMind** social sentiment across three platforms. XPOZ provides separate tools for each:

| Tool | Platform | Returns |
|---|---|---|
| `countTweets` | Twitter | Total volume |
| `getTwitterPostsByKeywords` | Twitter | Post text, author, engagement |
| `getRedditPostsByKeywords` | Reddit | Title, selftext, score |
| `getInstagramPostsByKeywords` | Instagram | Caption, username, likes |


In [12]:
count_r = await call_xpoz("countTweets", {
    "phrase": "Google DeepMind", "startDate": days_ago(7),
})
tweet_count = count_r["data"].get("count", 0)
print(f"7-day Google DeepMind tweet volume: {tweet_count:,}")


7-day Google DeepMind tweet volume: 77,003


In [13]:
# Two batches to get ~600 tweets (API caps at 300 per call)
tw1 = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "\"Google DeepMind\" OR DeepMind", "startDate": days_ago(3), "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"],
})
tw2 = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "\"Google DeepMind\" OR DeepMind", "startDate": days_ago(7), "endDate": days_ago(3), "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"],
})
tweets = tw1["data"].get("results", []) + tw2["data"].get("results", [])
print(f"Twitter: {len(tweets)} posts")


Twitter: 600 posts


In [14]:
rd_r = await call_xpoz("getRedditPostsByKeywords", {
    "query": "\"Google DeepMind\" OR DeepMind", "startDate": days_ago(7), "limit": 300,
    "fields": ["title", "selftext", "authorUsername", "score", "subredditName", "createdAtDate"],
})
reddit_posts = rd_r["data"].get("results", [])
print(f"Reddit: {len(reddit_posts)} posts")


Reddit: 4 posts


In [15]:
ig_r = await call_xpoz("getInstagramPostsByKeywords", {
    "query": "DeepMind", "limit": 100,
    "fields": ["caption", "username", "likeCount", "commentCount", "createdAtDate"],
})
ig_posts = ig_r["data"].get("results", [])
print(f"Instagram: {len(ig_posts)} posts")


Instagram: 100 posts


In [16]:
total_posts = len(tweets) + len(reddit_posts) + len(ig_posts)
print(f"Total: {total_posts} posts sampled from {tweet_count:,} indexed tweets")

Total: 704 posts sampled from 77,003 indexed tweets


## Analyze with Gemini

Feed the social data to Gemini and request a structured sentiment analysis.

In [17]:
def _safe_int(v):
    try:
        return int(v) if v else 0
    except (ValueError, TypeError):
        return 0

tw_text = "\n".join(
    f"@{p.get('authorUsername', '?')}: {p.get('text', '')} "
    f"[likes:{_safe_int(p.get('likeCount'))}, RTs:{_safe_int(p.get('retweetCount'))}]"
    for p in tweets[:200]
)
rd_text = "\n".join(
    f"r/{p.get('subredditName', '')} \u2014 {p.get('title', '')}: "
    f"{(p.get('selftext') or '')[:200]} [score:{_safe_int(p.get('score'))}]"
    for p in reddit_posts[:100]
)
ig_text = "\n".join(
    f"@{p.get('username', '?')}: {(p.get('caption') or '')[:200]} "
    f"[likes:{_safe_int(p.get('likeCount'))}]"
    for p in ig_posts[:50]
)

In [18]:
prompt = f"""
    Analyze the social-media sentiment for Google DeepMind based on these posts from the last 7 days.
    Total tweet volume: {tweet_count:,}

    === Twitter ({len(tweets)} posts) ===
    {tw_text}

    === Reddit ({len(reddit_posts)} posts) ===
    {rd_text}

    === Instagram ({len(ig_posts)} posts) ===
    {ig_text}

    Return your analysis in EXACTLY this format:

    OVERALL SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

    KEY THEMES:
    1. <theme> — <sentiment> — <explanation>
    2. …
    3. …
    4. …
    5. …

    RISK SIGNALS:
    - <risk>
    - …

    OPPORTUNITIES:
    - <opportunity>
    - …

    TOP POSTS:
    1. <platform> @<author>: <quote> — <why it matters>
    2. …
    3. …
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)
analysis = response.text
print(analysis)


OVERALL SENTIMENT: Bullish (72%)

KEY THEMES:
1. Labor Revolt & Unionization — Bearish — 98% of Google DeepMind’s UK workforce voted to unionize, specifically protesting AI contracts with the US military and Israel. This represents the first major labor pushback at a frontier AI lab, signaling potential internal friction regarding ethics and defense-sector revenue.
2. Scientific Breakthroughs (AlphaGenome/AlphaEvolve) — Bullish — DeepMind continues to lead in "AI for Science." The unveiling of AlphaGenome (genetics) and AlphaEvolve (mathematics) reinforces its brand as a creator of specialized, high-reasoning intelligence rather than just conversational chatbots.
3. Federal Oversight & Compliance — Neutral — DeepMind, alongside Microsoft and xAI, signed agreements with NIST’s CAISI to allow the US government pre-deployment access to "bare" models for security testing. This is seen by investors as "responsible innovation" but by some critics as a loss of autonomy.
4. Multimedia & Creati

## Results

Look at the highest-engagement posts that fed into the analysis.

In [19]:
print("Top tweets by engagement:\n")
for t in sorted(tweets, key=lambda x: _safe_int(x.get("likeCount")), reverse=True)[:10]:
    likes = _safe_int(t.get("likeCount"))
    rts = _safe_int(t.get("retweetCount"))
    text = (t.get("text") or "")[:120]
    print(f"  @{t.get('authorUsername', '?')} ({likes:,} likes, {rts:,} RTs)")
    print(f"    {text}")
    print()

Top tweets by engagement:

  @shimon8282 (2,394 likes, 48 RTs)
    Major personal news: After 6 years, I am leaving Waymo to lead a new multi-agent learning team at DeepMind.

  @haileyhmt (1,835 likes, 21 RTs)
    got reply from jeff dean (chief scientist in google deepmind) that my research “sounds like an important problem”\n\nit 

  @superai_conf (1,297 likes, 383 RTs)
    OpenAI. Anthropic. Google DeepMind. Mistral AI. Cerebras. On stage with Balaji Srinivasan, Benedict Evans, Max Tegmark, 

  @caleb_friesen (1,063 likes, 113 RTs)
    Insane.\n\nA 26-year-old from Chandigarh just got a paper accepted at ICML. As a solo independent researcher. From India

  @CyberRobooo (1,004 likes, 142 RTs)
    Boston Dynamics just had a brutal C-suite shakeup \n\nCEO Robert Playter retired in February, the COO and CSO bounced ri

  @Yuchenj_UW (470 likes, 11 RTs)
    Google DeepMind researchers who are training the Gemini coding model just fell to their knees. https://t.co/WGvG0WiqBe

  @DeryaTR

In [20]:
print("Top Reddit posts by score:\n")
for p in sorted(reddit_posts, key=lambda x: _safe_int(x.get("score")), reverse=True)[:5]:
    score = _safe_int(p.get("score"))
    print(f"  r/{p.get('subredditName', '?')} \u2014 {p.get('title', '')} (score: {score:,})")
    print()

Top Reddit posts by score:

  r/stocks — Are hyperscalers turning into a winner take most market? Should I buy more $GOOGL or diversify? (score: 120)

  r/dataisbeautiful — [OC] My data visualization on my website https://the8088.com/news.html looking at what sources bring the most significance. (score: 0)

  r/conspiracy — Can You Trust It? (score: 0)

  r/OpenAI — List of people at big-tech / professors / researchers who've jumped shit to launch their own AI labs for something Frontier/Foundational/AGI/Superintelligence/WorldModel (score: 0)



## Next steps

This notebook demonstrated fetching cross-platform social data via XPOZ MCP and analyzing it with Gemini. You can extend this pattern to:

- Add TikTok data via `getTiktokPostsByKeywords`
- Track sentiment over time by running daily
- Compare brands with parallel queries (e.g., DeepMind vs OpenAI)
- Use [function calling](https://ai.google.dev/gemini-api/docs/function-calling) to let Gemini call XPOZ tools directly

See the [XPOZ documentation](https://xpoz.ai) for the full list of available MCP tools.